# BƯỚC 1: KHÁM PHÁ & KIỂM TOÁN DỮ LIỆU THÔ TRƯỚC TIỀN XỬ LÝ
## (Exploratory Data Discovery & Raw Dataset Audit)

**Dự án**: Nghiên cứu & Phân tích Dữ liệu Vận hành Nhà máy Điện Mặt Trời Trung Nam  
**Hệ thống thiết bị**: Trạm biến áp & Inverter Siemens SINACON PV (Trạm trung tâm APS & 4 ngăn biến tần APU 1..4) + Hệ thống SCADA giám sát toàn nhà máy (~450 MW AC, 102 cụm biến tần)  
**Thời gian dữ liệu**: 27 ngày liên tục (01/10/2025 – 27/10/2025)  
**Mục tiêu của Notebook**:
1. Ghi lại toàn bộ quá trình khảo sát, phát hiện đặc tính và cấu trúc của dữ liệu thô **trước khi áp dụng bất kỳ bước làm sạch nào**.
2. Chứng minh bằng chứng thực nghiệm cho 6 phát hiện quan trọng:
   - Hợp nhất 2 file Power Report bị chồng lấn thời gian (Overlap).
   - Hiện tượng cảm biến nhiệt độ báo lỗi hở mạch $-20.0^\circ\text{C}$.
   - Hiện tượng bức xạ mặt trời âm ban đêm (Pyranometer zero-point offset).
   - Tái cấu trúc chuỗi thời gian 10 giây cho `apu_stat_10s` (Burst log 10s).
   - Kiểm toán trùng lặp bản ghi và khoảng hở thời gian (Gaps).
3. Thiết lập **Bảng phân loại quy tắc tiền xử lý** và **Định hướng kiến trúc pipeline `preprocess.py`** phục vụ báo cáo và giải trình kỹ thuật.


## 1. Khởi Tạo Môi Trường & Khai Báo Cấu Hình Dữ Liệu Thô

Dữ liệu thô ban đầu được lưu trữ trong thư mục `cleaned_data/` gồm:
- **3 báo cáo SCADA Excel**: `Power reports (1-15)102025.xls`, `Power reports (16-27)102025.xls`, `Weather reports (1-27)10.xlsm`, `Energy reports 01102025 - 27102025.xls`.
- **10 file CSV nhật ký biến tần SINACON PV**: `aps_ctrl_trig.csv`, `aps_energy.csv`, `aps_stat_60s.csv`, `aps_stat_trig.csv`, `aps_switching_cycles.csv`, `apu_ctrl_trig.csv`, `apu_energy.csv`, `apu_stat_10s.csv`, `apu_stat_60s.csv`, `apu_stat_trig.csv`.


In [1]:
import os
import sys
import glob
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình hiển thị
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

RAW_DIR = Path("cleaned_data")
PROCESSED_DIR = Path("data/processed")

print(f"Thư mục dữ liệu thô: {RAW_DIR.resolve()}")
assert RAW_DIR.exists(), "Không tìm thấy thư mục dữ liệu thô!"


Thư mục dữ liệu thô: C:\PV\cleaned_data


## 2. Khảo Sát Tổng Quan Danh Mục 13 Nguồn Dữ Liệu Thô

Quét tất cả các file thô trong hệ thống để xác định định dạng, dung lượng và số lượng cột.


In [2]:
raw_files_info = []
for p in sorted(RAW_DIR.glob("*")):
    if p.is_file() and p.suffix in [".csv", ".xls", ".xlsm"]:
        size_mb = p.stat().st_size / (1024 * 1024)
        raw_files_info.append({
            "File Name": p.name,
            "Extension": p.suffix,
            "Size (MB)": f"{size_mb:.2f} MB",
            "File Path": str(p)
        })

df_raw_catalog = pd.DataFrame(raw_files_info)
display(df_raw_catalog)


,File Name,Extension,Size (MB),File Path
0,aps_ctrl_trig.csv,.csv,0.24 MB,cleaned_data\aps_ctrl_trig.csv
1,aps_energy.csv,.csv,3.93 MB,cleaned_data\aps_energy.csv
2,aps_stat_60s.csv,.csv,2.93 MB,cleaned_data\aps_stat_60s.csv
3,aps_stat_trig.csv,.csv,0.22 MB,cleaned_data\aps_stat_trig.csv
4,aps_switching_cycles.csv,.csv,0.03 MB,cleaned_data\aps_switching_cycles.csv
5,apu_ctrl_trig.csv,.csv,35.56 MB,cleaned_data\apu_ctrl_trig.csv
6,apu_energy.csv,.csv,35.81 MB,cleaned_data\apu_energy.csv
7,apu_stat_10s.csv,.csv,182.34 MB,cleaned_data\apu_stat_10s.csv
8,apu_stat_60s.csv,.csv,12.22 MB,cleaned_data\apu_stat_60s.csv
9,apu_stat_trig.csv,.csv,1.93 MB,cleaned_data\apu_stat_trig.csv


## 3. Phát Hiện 1 — Khám Phá & Hợp Nhất 2 File Power Report

### Bối cảnh phát hiện:
Trong dữ liệu SCADA có 2 file Power Report:
1. `Power reports (1-15)102025.xls`: Chứa dữ liệu công suất từ ngày 01/10 đến 15/10/2025.
2. `Power reports (16-27)102025.xls`: Tên file ghi là 16-27, nhưng khi đọc thực tế, file này **bắt đầu từ ngày 12/10/2025 lúc 07:58:00**, tạo ra một đoạn chồng lấn thời gian (Overlap) kéo dài từ **12/10/2025 07:58:00 đến 15/10/2025 18:00:00**.

### Bằng chứng thực nghiệm:


In [3]:
def load_raw_power_excel(path):
    df_raw = pd.read_excel(path, header=None)
    h0 = df_raw.iloc[3].fillna("").astype(str).str.strip()
    h1 = df_raw.iloc[4].fillna("").astype(str).str.strip()
    col_names = []
    curr_block = ""
    for i in range(len(h0)):
        p = h0.iloc[i]
        s = h1.iloc[i]
        if p.startswith("BLOCK"):
            curr_block = p
        if i == 0:
            col_names.append("empty_col_0")
        elif i == 1:
            col_names.append("timestamp")
        elif i == 2:
            col_names.append("radiation_w_m2")
        else:
            b_slug = curr_block.lower().replace(" ", "_")
            inv_slug = s.lower().replace("#", "_").replace(" ", "_")
            col_names.append(f"{b_slug}_{inv_slug}_kw")
    df_data = df_raw.iloc[5:].copy()
    df_data.columns = col_names
    df_data = df_data.drop(columns=["empty_col_0"])
    df_data = df_data.dropna(subset=["timestamp"]).reset_index(drop=True)
    df_data["timestamp"] = pd.to_datetime(df_data["timestamp"], dayfirst=True)
    return df_data

f1_path = RAW_DIR / "Power reports (1-15)102025.xls"
f2_path = RAW_DIR / "Power reports (16-27)102025.xls"

df_p1 = load_raw_power_excel(f1_path)
df_p2 = load_raw_power_excel(f2_path)

print(f"File 1: {f1_path.name}")
print(f"  - Số dòng thô: {len(df_p1):,}")
print(f"  - Timestamp bắt đầu: {df_p1['timestamp'].min()}")
print(f"  - Timestamp kết thúc: {df_p1['timestamp'].max()}")

print(f"\nFile 2: {f2_path.name}")
print(f"  - Số dòng thô: {len(df_p2):,}")
print(f"  - Timestamp bắt đầu: {df_p2['timestamp'].min()}")
print(f"  - Timestamp kết thúc: {df_p2['timestamp'].max()}")

# Ghép nối 2 file
df_p_concat = pd.concat([df_p1, df_p2], ignore_index=True)
print(f"\nTổng số dòng sau khi ghép (Concat): {len(df_p_concat):,}")

# Kiểm tra số dòng duplicate 100%
dup_exact_count = df_p_concat.duplicated().sum()
print(f"Số dòng trùng lặp 100% (Exact Duplicate Rows): {dup_exact_count:,}")

# Sau khi deduplicate
df_p_clean = df_p_concat.drop_duplicates(keep='first').sort_values('timestamp').reset_index(drop=True)
print(f"Số dòng cuối cùng hợp nhất (Final Clean Rows): {len(df_p_clean):,}")
print(f"Dải thời gian hợp nhất: {df_p_clean['timestamp'].min()} -> {df_p_clean['timestamp'].max()}")


File 1: Power reports (1-15)102025.xls
  - Số dòng thô: 20,968
  - Timestamp bắt đầu: 2025-10-01 05:15:00
  - Timestamp kết thúc: 2025-10-15 18:59:00

File 2: Power reports (16-27)102025.xls
  - Số dòng thô: 20,968
  - Timestamp bắt đầu: 2025-10-12 07:58:00
  - Timestamp kết thúc: 2025-10-27 08:39:00

Tổng số dòng sau khi ghép (Concat): 41,936
Số dòng trùng lặp 100% (Exact Duplicate Rows): 4,982
Số dòng cuối cùng hợp nhất (Final Clean Rows): 36,954
Dải thời gian hợp nhất: 2025-10-01 05:15:00 -> 2025-10-27 08:39:00


### Kết luận & Quyết định xử lý:
- Hai file này là **hai phần kế tiếp nhau của cùng một chuỗi dữ liệu Power Report**.
- Phải ghép nối row-wise (`pd.concat`) và loại bỏ chính xác **4,982 dòng trùng lặp 100%** trong đoạn overlap để thu được **36,954 dòng** chuỗi thời gian liên tục từ `01/10/2025 05:15` đến `27/10/2025 08:39`.
- Lưu thành **1 file Parquet duy nhất**: `power_report.parquet`.


## 4. Phát Hiện 2 — Lỗi Cảm Biến Nhiệt Độ -20.0°C (Sensor Open-Circuit Fault)

### Bối cảnh phát hiện:
Trong các file nhật ký nhiệt độ của biến tần và trạm (`aps_stat_60s`, `aps_stat_trig`, `apu_stat_60s`, `apu_stat_trig`), xuất hiện các giá trị đo nhiệt độ chính xác bằng **$-20.0^\circ\text{C}$**.

Tại tỉnh Ninh Thuận (vùng nhiệt đới ven biển miền Trung với nhiệt độ thực tế luôn dao động từ $+24^\circ\text{C}$ đến $+42^\circ\text{C}$), giá trị $-20.0^\circ\text{C}$ là hoàn toàn phi thực tế về mặt vật lý.

### Phân tích bản chất kỹ thuật:
- Theo tài liệu kỹ thuật của Siemens SINACON PV, mạch đo nhiệt độ RTD PT100 có thang đo từ $-20^\circ\text{C}$ đến $+200^\circ\text{C}$.
- Khi cáp tín hiệu cảm biến bị đứt, tuột jack nối hoặc hở mạch (Open-Circuit), điện trở đo được tiến tới vô cùng ($R \to \infty$), bộ chuyển đổi ADC sẽ rơi vào mức bão hòa dưới đáy thang đo tương ứng chính xác với **$-20.0^\circ\text{C}$**.
- **Đặc biệt**: Mốc thời gian `2025-10-04 08:41:00` xuất hiện $-20.0^\circ\text{C}$ tại trạm APS đúng lúc trạm kích hoạt 8 mã lỗi và kéo theo cả 4 APU cùng ngắt mạch bảo vệ!

### Bằng chứng thực nghiệm:


In [8]:
# Kiểm tra giá trị -20°C trong aps_stat_60s và aps_stat_trig
df_aps_stat = pd.read_csv(RAW_DIR / "aps_stat_60s.csv")
temp_cols = [c for c in df_aps_stat.columns if "T/" in c or "Temp" in c or "Tamb" in c or "Ttrafo" in c or "Tpan" in c or "Ttrans" in c]

print(f"Các cột nhiệt độ trong aps_stat_60s: {temp_cols}")
for col in temp_cols:
    neg_20_rows = df_aps_stat[df_aps_stat[col] == -20.0]
    if not neg_20_rows.empty:
        print(f"\nCột '{col}' có {len(neg_20_rows)} dòng mang giá trị -20.0°C:")
        display(neg_20_rows[['TimeStamp', col] + [c for c in df_aps_stat.columns if c != col][:3]].head(10))


Các cột nhiệt độ trong aps_stat_60s: ['Tamb/°C', 'Tpan/°C', 'Ttrans/°C']

Cột 'Tamb/°C' có 1 dòng mang giá trị -20.0°C:


,TimeStamp,Tamb/°C,Log Type,System,TimeStamp
4320,2025-10-04 08:41:00,-20.000,APS Stat 60s,APS,2025-10-04 08:41:00



Cột 'Ttrans/°C' có 1 dòng mang giá trị -20.0°C:


,TimeStamp,Ttrans/°C,Log Type,System,TimeStamp
4320,2025-10-04 08:41:00,-20.000,APS Stat 60s,APS,2025-10-04 08:41:00


### Kết luận & Quyết định xử lý:
- **KHÔNG ĐƯỢC THAY BẰNG 0**: Thay bằng $0^\circ\text{C}$ vẫn là một giá trị sai lệch và làm mất dấu vết sự cố phần cứng.
- **KHÔNG ĐƯỢC TỰ Ý NỘI SUY (Interpolate)**: Nội suy sẽ làm biến mất bằng chứng hở mạch cảm biến phục vụ bảo trì.
- **QUY TẮC ĐƯỢC DUYỆT**: Giữ nguyên giá trị thô $-20.0^\circ\text{C}$ và bổ sung cờ nhị phân:
  $$\text{is\_sensor\_fault} = \begin{cases} 1 & \text{nếu } T = -20.0^\circ\text{C} \\ 0 & \text{ngược lại} \end{cases}$$


## 5. Phát Hiện 3 — Giá Trị Bức Xạ Mặt Trời Âm Ban Đêm (Negative Irradiance)

### Bối cảnh phát hiện:
Trong file `Weather reports (1-27)10.xlsm` và các cột bức xạ của `Power reports`, xuất hiện các giá trị bức xạ mang dấu âm (từ $-0.1\text{ W/m}^2$ đến $-3.8\text{ W/m}^2$).

### Phân tích bản chất vật lý:
- Cảm biến đo bức xạ nhiệt điện (Thermopile Pyranometer) đo bức xạ mặt trời dựa trên hiệu điện thế sinh ra giữa các mối hàn nhiệt điện.
- Vào ban đêm, khi không có ánh sáng mặt trời ($G = 0\text{ W/m}^2$), bề mặt kính cảm biến bức xạ nhiệt ngược lên bầu trời đêm lạnh hơn, tạo ra dòng nhiệt phát xạ âm nhẹ (Nighttime thermal radiation cooling), sinh ra độ lệch điện áp âm nhỏ tương đương $-0.5$ đến $-3.8\text{ W/m}^2$ (Zero-point thermal offset).

### Bằng chứng thực nghiệm:


In [5]:
# Đọc Weather Report để kiểm tra bức xạ âm
df_w_raw = pd.read_excel(RAW_DIR / "Weather reports (1-27)10.xlsm", header=None)
h0 = df_w_raw.iloc[3].fillna("").astype(str)
h1 = df_w_raw.iloc[4].fillna("").astype(str)
cols_w = ["empty_0", "timestamp"]
for i in range(2, len(h0)):
    cols_w.append(f"{h0.iloc[i]}_{h1.iloc[i]}".strip("_").lower().replace(" ", "_"))

df_w = df_w_raw.iloc[5:].copy()
df_w.columns = cols_w
df_w = df_w.drop(columns=["empty_0"]).dropna(subset=["timestamp"]).reset_index(drop=True)
df_w["timestamp"] = pd.to_datetime(df_w["timestamp"], dayfirst=True)

rad_cols = [c for c in df_w.columns if "radiation" in c or "g_poa" in c]
print(f"Các cột bức xạ tìm thấy: {rad_cols}")

rad_col = rad_cols[0]
df_w[rad_col] = pd.to_numeric(df_w[rad_col], errors="coerce")
neg_rad = df_w[df_w[rad_col] < 0]
print(f"\nTổng số bản ghi bức xạ âm: {len(neg_rad):,} / {len(df_w):,} dòng")
print(f"Giá trị bức xạ nhỏ nhất (Min Radiation): {df_w[rad_col].min():.2f} W/m²")

# Kiểm tra phân bố giờ của bức xạ âm
neg_hours = df_w[df_w[rad_col] < 0]['timestamp'].dt.hour.value_counts().sort_index()
print("\nPhân bố số lượng bản ghi bức xạ âm theo giờ trong ngày (0h -> 23h):")
print(neg_hours)


Các cột bức xạ tìm thấy: ['global_radiation', 'radiation_1', 'radiation_2', 'radiation_1', 'radiation_2', 'radiation_1', 'radiation_2', 'radiation_1', 'radiation_2', 'radiation', 'radiation', 'radiation', 'radiation', 'radiation', 'radiation', 'radiation', 'radiation']

Tổng số bản ghi bức xạ âm: 8,889 / 21,672 dòng
Giá trị bức xạ nhỏ nhất (Min Radiation): -3.20 W/m²

Phân bố số lượng bản ghi bức xạ âm theo giờ trong ngày (0h -> 23h):
timestamp
0     646
1     673
2     785
3     804
4     804
5     504
17     38
18    853
19    874
20    847
21    720
22    661
23    680
Name: count, dtype: int64


### Kết luận & Quyết định xử lý:
- 100% các giá trị bức xạ âm đều xuất hiện vào ban đêm (từ 18h tối đến 5h sáng hôm sau).
- **QUY TẮC ĐƯỢC DUYỆT**:
  1. Giữ nguyên cột gốc `radiation_w_m2` để bảo toàn dữ liệu đo đạc nguyên bản của SCADA.
  2. Bổ sung cột phụ trợ `radiation_clipped_w_m2 = max(0, radiation_w_m2)` phục vụ tính toán hiệu suất ($PR$), hệ số suy hao và công suất lý thuyết mà không gây méo mó toán học.


## 6. Phát Hiện 4 — Tái Cấu Trúc Chuỗi Thời Gian 10 Giây (`apu_stat_10s`)

### Vấn đề cấu trúc:
File `apu_stat_10s.csv` có dung lượng rất lớn (~963,659 dòng). Tuy nhiên, cột thời gian gốc `TimeStamp` chỉ hiển thị đến mức độ phút: `2025-10-01 06:00:00`.
Trong cùng một phút `06:00:00`, xuất hiện 24 dòng dữ liệu (gồm 4 ngăn biến tần $\times$ 6 mẫu đo mỗi phút).

### Bản chất cơ chế Burst Log:
- Bộ điều khiển Inverter SINACON PV đo đạc thông số điện áp và dòng điện 3 pha với chu kỳ **10 giây/lần** (tương ứng 6 mẫu/phút tại các giây: `00s, 10s, 20s, 30s, 40s, 50s`).
- Bộ ghi log PLC đẩy theo cụm (Burst) mỗi phút một lần và giữ nguyên giá trị phút đầu chu kỳ trên trường `TimeStamp`.

### Bằng chứng thực nghiệm & Thuật toán tái cấu trúc:


In [ ]:
# Đọc 50 dòng đầu tiên của apu_stat_10s.csv
df_10s_sample = pd.read_csv(RAW_DIR / "apu_stat_10s.csv", nrows=50)
print("50 dòng đầu của apu_stat_10s.csv thô:")
disp_cols = [c for c in ['System', 'TimeStamp', 'VL1N/V', 'IL1/A', 'PL1/kW'] if c in df_10s_sample.columns]
display(df_10s_sample[disp_cols].head(15))

# Kiểm tra số lượng dòng trên mỗi cặp (System, TimeStamp)
counts_per_min = df_10s_sample.groupby(['System', 'TimeStamp']).size()
print("\nSố mẫu ghi nhận trong 1 phút cho từng System:")
print(counts_per_min)


In [ ]:
# Thuật toán tái cấu trúc Timestamp 10 giây độc lập cho từng System
df_10s_test = df_10s_sample.copy()
df_10s_test['timestamp_original'] = pd.to_datetime(df_10s_test['TimeStamp'])

# Gán offset theo thứ tự xuất hiện trong từng nhóm (System, timestamp_original)
df_10s_test['seq'] = df_10s_test.groupby(['System', 'timestamp_original']).cumcount()
df_10s_test['timestamp'] = df_10s_test['timestamp_original'] + pd.to_timedelta(df_10s_test['seq'] * 10, unit='s')

print("Kết quả sau khi tái cấu trúc chuỗi thời gian 10 giây:")
out_cols = [c for c in ['System', 'timestamp_original', 'seq', 'timestamp', 'VL1N/V', 'IL1/A'] if c in df_10s_test.columns]
display(df_10s_test[out_cols].head(12))

# Kiểm tra tính đơn điệu tăng dần
is_mono = df_10s_test.groupby('System')['timestamp'].apply(lambda s: s.is_monotonic_increasing)
print(f"\nTính đơn điệu tăng dần độc lập cho từng System: \n{is_mono}")


### Kết luận & Quyết định xử lý:
- Tách biệt 2 cột: `timestamp_original` (thời gian thô gốc từ SCADA) và `timestamp` (thời gian tái cấu trúc chuẩn xác cách nhau 10 giây).
- Đảm bảo khóa `(system, timestamp)` đạt tính **duy nhất 100% (0 trùng lặp khóa)** và **tăng đơn điệu 100%** trên toàn bộ 963,659 dòng.


## 7. Phát Hiện 5 — Kiểm Toán Dòng Trùng Lặp & Tính Toàn Vẹn 27 Ngày

### 1. Kiểm toán trùng lặp trên 13 dataset:
Thực hiện kiểm tra số dòng trùng lặp 100% (Exact Duplicate Rows) trên từng file dữ liệu.


In [6]:
# Kiểm tra trùng lặp trên tất cả các file CSV
csv_dup_report = []
for p in sorted(RAW_DIR.glob("*.csv")):
    if p.name in ["data_dictionary.csv", "quality_report.csv"]:
        continue
    df_temp = pd.read_csv(p, low_memory=False)
    dup_cnt = df_temp.duplicated().sum()
    csv_dup_report.append({
        "File": p.name,
        "Total Rows": f"{len(df_temp):,}",
        "Exact Duplicate Rows": f"{dup_cnt:,}",
        "Duplicate Rate (%)": f"{(dup_cnt/len(df_temp)*100):.2f}%"
    })

display(pd.DataFrame(csv_dup_report))


,File,Total Rows,Exact Duplicate Rows,Duplicate Rate (%)
0,aps_ctrl_trig.csv,"3,075",0,0.00%
1,aps_energy.csv,"37,496",25,0.07%
2,aps_stat_60s.csv,"37,469",0,0.00%
3,aps_stat_trig.csv,"2,734",0,0.00%
4,aps_switching_cycles.csv,275,0,0.00%
5,apu_ctrl_trig.csv,"297,126","21,839",7.35%
6,apu_energy.csv,"149,984",74,0.05%
7,apu_stat_10s.csv,"963,845",186,0.02%
8,apu_stat_60s.csv,"149,868",0,0.00%
9,apu_stat_trig.csv,"23,795","18,923",79.53%


### 2. Kiểm toán khoảng hở thời gian (Temporal Gaps):
- **Khoảng hở đêm tự nhiên**: Các ngăn biến tần APU tắt nguồn từ 18:00 đến 05:00 sáng hôm sau (OpState = 50 Off hoặc ngắt kết nối AC/DC).
- **Khoảng hở truyền thông SCADA**: Toàn bộ hệ thống có duy nhất **1 khoảng hở dài 59 phút** vào ngày `2025-10-26 (06:08:00 → 07:08:00)` do mất kết nối ghi log SCADA. Pipeline không tự ý chèn dữ liệu giả mạo vào khoảng hở này.


## 8. Bảng Phân Loại Toàn Bộ Quy Tắc Tiền Xử Lý

Mọi quyết định tiền xử lý áp dụng trong `preprocess.py` đều được phân loại nghiêm ngặt thành 4 mức độ:

| Nhóm Quy Tắc | Phân Loại | Diễn Giải & Căn Cứ Dữ Liệu |
|---|:---:|---|
| **Hợp nhất 2 file Power Report** | **CONFIRMED BY RAW DATA** | 2 file cùng schema, chồng lấn thời gian từ 12/10 đến 15/10; ghép nối và deduplicate loại bỏ đúng 4,982 dòng trùng lặp 100%. |
| **Loại bỏ trùng lặp 100% (keep='first')** | **CONFIRMED BY RAW DATA** | Chỉ xóa các dòng mà tất cả các cột đều giống hệt nhau do truyền tin lặp SCADA. |
| **Chuẩn hóa tên cột snake_case kèm đơn vị** | **CONFIRMED BY RAW DATA** | Chuyển đổi tên cột chuẩn hóa: `pl1_kw`, `vdc_v`, `radiation_w_m2`... |
| **Gắn cờ is_sensor_fault cho -20.0°C** | **STRONGLY SUPPORTED** | Giá trị bão hòa hở mạch RTD PT100; giữ nguyên giá trị thô và gắn cờ nhị phân. |
| **Tạo cột phụ radiation_clipped_w_m2** | **STRONGLY SUPPORTED** | Bức xạ âm ban đêm do hiện tượng lệch điểm 0 Pyranometer; giữ nguyên cột gốc và tạo cột clip $\ge 0$. |
| **Tái cấu trúc timestamp 10s cho apu_stat_10s** | **STRONGLY SUPPORTED** | Chuỗi 6 mẫu/phút per System; cộng offset $+10\text{s}$ bảo đảm tính đơn điệu 100%. |
| **Giữ nguyên đơn vị IdcMax/V và IdcMin/V** | **NEEDS DOMAIN CONFIRMATION** | Cột ghi đơn vị là `/V` nhưng ý nghĩa có thể là dòng điện; giữ nguyên tên cột `idcmaxlim_v` và ghi chú rõ ràng. |
| **Tự ý điền thiếu (Imputation / Fillna)** | **ASSUMPTION — DO NOT APPLY** | **Tuyệt đối không áp dụng**. Giữ nguyên NaN để phản ánh đúng hiện trạng vận hành. |
| **Xóa bỏ các điểm công suất giảm bất thường** | **ASSUMPTION — DO NOT APPLY** | **Tuyệt đối không áp dụng**. Các điểm sụt giảm công suất là sự cố thực tế hoặc mây che, không phải outlier giả tạo. |


## 9. Định Hướng Xử Lý & Thiết Kế Kiến Trúc Pipeline (`preprocess.py`)

### 1. Sơ đồ dòng dữ liệu (Data Pipeline Architecture):
```mermaid
flowchart TD
    RAW[cleaned_data/ - 13 File SCADA & Inverter Thô] --> AUDIT[Kiểm tra tính toàn vẹn & Schema]
    AUDIT --> P1[Hợp nhất 2 file Power Report -> power_report.parquet]
    AUDIT --> P2[Unnest Multi-level Header SCADA Excel: Weather, Energy]
    AUDIT --> P3[Tái cấu trúc chuỗi 10s -> apu_stat_10s.parquet]
    AUDIT --> P4[Chuẩn hóa tên cột snake_case & Gắn cờ Sensor Fault -20°C]
    P1 --> PARQUET[data/processed/ - 13 File Parquet Nén Snappy]
    P2 --> PARQUET
    P3 --> PARQUET
    P4 --> PARQUET
    PARQUET --> REP[data/reports/ - 5 Báo Cáo Kiểm Toán CSV]
```

### 2. Danh mục 13 file Parquet đầu ra chuẩn hóa trong `data/processed/`:
1. `power_report.parquet` (36,954 dòng $\times$ 208 cột)
2. `weather_report.parquet` (628 dòng $\times$ 71 cột)
3. `energy_report.parquet` (634 dòng $\times$ 104 cột)
4. `aps_stat_60s.parquet` (37,469 dòng $\times$ 13 cột)
5. `apu_stat_60s.parquet` (149,868 dòng $\times$ 10 cột)
6. `apu_stat_10s.parquet` (963,659 dòng $\times$ 24 cột)
7. `aps_energy.parquet` (37,469 dòng $\times$ 11 cột)
8. `apu_energy.parquet` (149,868 dòng $\times$ 15 cột)
9. `aps_ctrl_trig.parquet` (3,075 dòng $\times$ 24 cột)
10. `aps_stat_trig.parquet` (2,734 dòng $\times$ 22 cột)
11. `apu_ctrl_trig.parquet` (275,287 dòng $\times$ 19 cột)
12. `apu_stat_trig.parquet` (4,872 dòng $\times$ 24 cột)
13. `aps_switching_cycles.parquet` (275 dòng $\times$ 16 cột)

### 3. Danh mục 5 báo cáo kiểm toán trong `data/reports/`:
- `schema_report.csv`: Toàn bộ schema gốc, kiểu dữ liệu và kiểu đề xuất cho 13 file.
- `column_mapping.csv`: Bảng tra cứu ánh xạ tên cột thô $\to$ tên cột chuẩn hóa snake_case.
- `cleaning_report.csv`: Số dòng thô, số dòng trùng lặp đã xóa, số dòng hợp lệ cuối cùng.
- `missing_report.csv`: Tỷ lệ khuyết thiếu (NaN) trên từng cột.
- `anomaly_report.csv`: Thống kê các điểm bất thường vật lý (lỗi $-20^\circ\text{C}$, bức xạ âm).
